In [1]:
from pyspark.sql import SparkSession
from delta import *

builder = SparkSession.builder.appName("DeltaLake_SATC") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.warehouse.dir", "../data/warehouse") # Onde os dados ficam

spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/04/18 14:36:15 WARN Utils: Your hostname, PC-Octavio resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/18 14:36:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/source/trabalho1ed/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/octavio/.ivy2/cache
The jars for the packages stored in: /home/octavio/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cd3d3a52-71ae-4a0c-8aa4-147c3da26856;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (502ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (258ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (580ms)
:: resolution report :: resolve 3510ms 

Implementação do Delta Lake: Ingestão de Dados
Nesta etapa, inicializamos a estrutura do projeto criando nossa primeira tabela no formato Delta Lake. A principal vantagem aqui é que os dados são armazenados como arquivos Parquet, mas com um log de transações que permite operações ACID (Atomicidade, Consistência, Isolamento e Durabilidade)

In [2]:
# 1. Criar dados manuais para teste
dados = [
    (1, "Notebook", "Eletrônicos", 4500.0, 1, "2026-04-18"),
    (2, "Mouse", "Acessórios", 120.0, 2, "2026-04-18")
]

# 2. Definir as colunas
colunas = ["id_venda", "produto", "categoria", "valor", "quantidade", "data"]

# 3. Transformar os dados em um DataFrame do Spark
df = spark.createDataFrame(dados, colunas)

# 4. Salvar os dados como uma Tabela Delta na pasta /data do projeto
# O "overwrite" garante que se você rodar de novo, ele sobrescreve sem erro
df.write.format("delta").mode("overwrite").save("../data/tabela_vendas_delta")

print("Tabela Delta criada com sucesso em /data/tabela_vendas_delta!")

Tabela Delta criada com sucesso em /data/tabela_vendas_delta!


Operação de UPDATE no Delta Lake
Diferente de formatos de arquivo tradicionais, o Delta Lake permite atualizações granulares. Utilizamos a API DeltaTable para localizar registros específicos através de condições e aplicar modificações sem a necessidade de reprocessar todo o conjunto de dados.

In [3]:
from delta.tables import *
from pyspark.sql.functions import *

# 1. Carregar a tabela Delta que acabamos de criar
tabela_delta = DeltaTable.forPath(spark, "../data/tabela_vendas_delta")

# 2. Vamos simular um reajuste de preço:
# "Aumentar o valor do Mouse (id_venda = 2) para 135.0"
print("Atualizando o preço do Mouse...")
tabela_delta.update(
    condition = expr("id_venda = 2"),
    set = { "valor": lit(135.0) }
)

# 3. Visualizar o resultado para confirmar
tabela_delta.toDF().show()

Atualizando o preço do Mouse...


26/04/18 14:59:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+--------+-----------+------+----------+----------+
|id_venda| produto|  categoria| valor|quantidade|      data|
+--------+--------+-----------+------+----------+----------+
|       1|Notebook|Eletrônicos|4500.0|         1|2026-04-18|
|       2|   Mouse| Acessórios| 135.0|         2|2026-04-18|
+--------+--------+-----------+------+----------+----------+



Operação de DELETE no Delta Lake
O comando delete remove registros da tabela que satisfaçam um predicado específico. No armazenamento físico, os arquivos antigos são marcados como removidos no log, mas permanecem no disco por um período (configurável), permitindo auditoria e recuperação de dados.

In [4]:
# 1. Vamos deletar a venda do Notebook (id_venda = 1)
print("Deletando a venda do Notebook...")
tabela_delta.delete(condition = expr("id_venda = 1"))

# 2. Verificar se sobrou apenas o Mouse (id_venda = 2)
tabela_delta.toDF().show()

Deletando a venda do Notebook...
+--------+-------+----------+-----+----------+----------+
|id_venda|produto| categoria|valor|quantidade|      data|
+--------+-------+----------+-----+----------+----------+
|       2|  Mouse|Acessórios|135.0|         2|2026-04-18|
+--------+-------+----------+-----+----------+----------+



In [5]:
# 1. Ver o histórico de transações da tabela
print("Histórico da Tabela:")
tabela_delta.history().select("version", "timestamp", "operation", "operationParameters").show()

# 2. Consultar a Versão 0 (Como a tabela era logo após o INSERT)
print("Consultando a Versão 0 (Dados Originais):")
df_versao_zero = spark.read.format("delta").option("versionAsOf", 0).load("../data/tabela_vendas_delta")
df_versao_zero.show()

Histórico da Tabela:
+-------+--------------------+---------+--------------------+
|version|           timestamp|operation| operationParameters|
+-------+--------------------+---------+--------------------+
|      2|2026-04-18 15:03:...|   DELETE|{predicate -> ["(...|
|      1|2026-04-18 14:59:...|   UPDATE|{predicate -> ["(...|
|      0|2026-04-18 14:57:...|    WRITE|{mode -> Overwrit...|
+-------+--------------------+---------+--------------------+

Consultando a Versão 0 (Dados Originais):
+--------+--------+-----------+------+----------+----------+
|id_venda| produto|  categoria| valor|quantidade|      data|
+--------+--------+-----------+------+----------+----------+
|       1|Notebook|Eletrônicos|4500.0|         1|2026-04-18|
|       2|   Mouse| Acessórios| 120.0|         2|2026-04-18|
+--------+--------+-----------+------+----------+----------+

